In [1]:
import os
import sys
import random
import subprocess

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

try:
    import numpy as np
    import pandas as pd
    import tensorflow as tf
except Exception:
    os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
    os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "3"
    for m in list(sys.modules.keys()):
        if m.startswith("google.protobuf"):
            del sys.modules[m]
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "protobuf==3.20.3"]
    )
    for m in list(sys.modules.keys()):
        if m.startswith("google.protobuf"):
            del sys.modules[m]
    import numpy as np
    import pandas as pd
    import tensorflow as tf

from tensorflow.keras.optimizers import Adam

print("TF version:", tf.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

try:
    tf.config.threading.set_intra_op_parallelism_threads(0)
    tf.config.threading.set_inter_op_parallelism_threads(0)
except Exception:
    pass




2026-05-12 18:39:38.011691: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778611178.044216      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778611178.050393      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TF version: 2.18.0


In [2]:
def processAndWriteDf(df):
    if "image_id" not in df.columns:
        raise ValueError("Expected 'image_id' column in dataframe.")
    df.to_csv("./submission.csv", index=False)
    print("write done -> ./submission.csv")
    return df




In [3]:
def _resolve_pp2020_base():
    """
    Fix: choose the real existing Kaggle input path (varies by environment).
    Supports both Kaggle's canonical /kaggle/input and the provided /kaggle/data layouts.
    """
    candidates = [
        "/kaggle/input/plant-pathology-2020-fgvc7",
        "/kaggle/data/plant-pathology-2020-fgvc7",
        "/kaggle/input",
        "/kaggle/data",
        "../input/plant-pathology-2020-fgvc7",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(
            os.path.join(c, "test.csv")
        ):
            return c

        nested = os.path.join(c, "plant-pathology-2020-fgvc7")
        if os.path.exists(os.path.join(nested, "train.csv")) and os.path.exists(
            os.path.join(nested, "test.csv")
        ):
            return nested

    raise FileNotFoundError(f"Could not find dataset base dir in: {candidates}")


def getPredictionFromTPUModel():
    base = _resolve_pp2020_base()
    base_dir = os.path.join(base, "images")
    train_path = os.path.join(base, "train.csv")
    test_path = os.path.join(base, "test.csv")
    sample_path = os.path.join(base, "sample_submission.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sample_sub = pd.read_csv(sample_path)

    label_cols = [c for c in sample_sub.columns if c != "image_id"]

    test_image_ids = test_df["image_id"].astype(str).to_numpy()

    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df["image_id"] = train_df["image_id"].astype(str) + ".jpg"
    test_df["image_id"] = test_df["image_id"].astype(str) + ".jpg"

    img_size = (320, 320)
    batch_size = 32
    epochs_stage1 = 18
    epochs_stage2 = 6

    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

    n = len(train_df)
    val_n = int(np.floor(0.1 * n))
    train_n = n - val_n
    train_part = train_df.iloc[:train_n]
    val_part = train_df.iloc[train_n:]

    train_files = (base_dir + "/" + train_part["image_id"]).to_numpy()
    val_files = (base_dir + "/" + val_part["image_id"]).to_numpy()
    test_files = (base_dir + "/" + test_df["image_id"]).to_numpy()

    y_train = train_part[label_cols].to_numpy(dtype=np.float32)
    y_val = val_part[label_cols].to_numpy(dtype=np.float32)

    # CHANGE (timeout): only compute seeds for the training set (the only place augmentation uses seeds).
    # This is equivalent because val/test seeds were never used (no augmentation there), so removing them saves time.
    def _make_seeds_np(files_np: np.ndarray) -> np.ndarray:
        files_tf = tf.constant(files_np.astype(str))
        h = tf.strings.to_hash_bucket_fast(files_tf, 2**31 - 1)
        s0 = tf.cast(tf.math.floormod(h, 2**16), tf.int32)
        s1 = tf.fill(tf.shape(s0), tf.cast(SEED, tf.int32))
        seeds_tf = tf.stack([s0, s1], axis=1)
        return seeds_tf.numpy()

    train_seeds = _make_seeds_np(train_files)

    AUTO = tf.data.AUTOTUNE

    def _decode_resize(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3, dct_method="INTEGER_FAST")
        img = tf.image.resize(
            img, img_size, method=tf.image.ResizeMethod.BILINEAR, antialias=False
        )
        img = preprocess_input(img)
        return img

    def _augment_no_tfa(img, seed):
        img = tf.image.stateless_random_flip_left_right(img, seed=seed)

        dx = tf.random.stateless_uniform(
            [], seed=seed + tf.constant([1, 0], tf.int32), minval=-0.06, maxval=0.06
        )
        dy = tf.random.stateless_uniform(
            [], seed=seed + tf.constant([0, 1], tf.int32), minval=-0.06, maxval=0.06
        )
        tx = tf.cast(tf.round(dx * tf.cast(img_size[1], tf.float32)), tf.int32)
        ty = tf.cast(tf.round(dy * tf.cast(img_size[0], tf.float32)), tf.int32)
        img = tf.roll(img, shift=[ty, tx], axis=[0, 1])

        z = tf.random.stateless_uniform(
            [],
            seed=seed + tf.constant([2, 2], tf.int32),
            minval=1.0 - 0.08,
            maxval=1.0 + 0.08,
        )
        new_h = tf.cast(tf.round(z * tf.cast(img_size[0], tf.float32)), tf.int32)
        new_w = tf.cast(tf.round(z * tf.cast(img_size[1], tf.float32)), tf.int32)
        img2 = tf.image.resize(
            img, [new_h, new_w], method=tf.image.ResizeMethod.BILINEAR, antialias=False
        )
        img2 = tf.image.resize_with_crop_or_pad(img2, img_size[0], img_size[1])

        angle = tf.random.stateless_uniform(
            [], seed=seed + tf.constant([3, 3], tf.int32), minval=-12.0, maxval=12.0
        ) * (np.pi / 180.0)
        cos_a = tf.math.cos(angle)
        sin_a = tf.math.sin(angle)
        cx = (img_size[1] - 1) / 2.0
        cy = (img_size[0] - 1) / 2.0

        a0 = cos_a
        a1 = -sin_a
        a2 = cx - cos_a * cx + sin_a * cy
        b0 = sin_a
        b1 = cos_a
        b2 = cy - sin_a * cx - cos_a * cy
        transform = tf.stack([a0, a1, a2, b0, b1, b2, 0.0, 0.0])[None, :]

        img2 = tf.raw_ops.ImageProjectiveTransformV3(
            images=img2[None, ...],
            transforms=transform,
            output_shape=tf.constant(img_size, dtype=tf.int32),
            interpolation="BILINEAR",
            fill_mode="NEAREST",
            fill_value=0.0,
        )[0]
        return img2

    def _cache_path_for(name: str) -> str:
        return os.path.join(
            "/kaggle/working", f"tfdata_cache_{name}_{img_size[0]}x{img_size[1]}"
        )

    # CHANGE (timeout): cache only decoded/resized images (pure function of file path).
    # Correctness preserved because augmentation still runs every epoch (after cache) with the same stateless seeds,
    # and labels are not modified. This avoids repeated JPEG decode/resize and avoids caching augmented variants.
    def make_image_ds(files, cache_name=None):
        opt = tf.data.Options()
        opt.experimental_deterministic = True
        ds = tf.data.Dataset.from_tensor_slices(files).with_options(opt)
        ds = ds.map(_decode_resize, num_parallel_calls=AUTO, deterministic=True)
        ds = ds.apply(tf.data.experimental.ignore_errors())
        if cache_name is not None:
            ds = ds.cache(_cache_path_for(cache_name))
        return ds

    def make_train_ds(files, labels, seeds, cache_name="train_images"):
        opt = tf.data.Options()
        opt.experimental_deterministic = True

        img_ds = make_image_ds(files, cache_name=cache_name)
        lab_ds = tf.data.Dataset.from_tensor_slices(labels)
        seed_ds = tf.data.Dataset.from_tensor_slices(seeds)

        ds = tf.data.Dataset.zip((img_ds, lab_ds, seed_ds)).with_options(opt)

        ds = ds.shuffle(
            buffer_size=min(
                2048, int(files.shape[0] if hasattr(files, "shape") else len(files))
            ),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

        def _apply_aug(img, y, seed):
            img = _augment_no_tfa(img, seed)
            return img, y

        ds = ds.map(_apply_aug, num_parallel_calls=AUTO, deterministic=True)
        ds = ds.batch(batch_size, drop_remainder=False)
        ds = ds.prefetch(AUTO)
        return ds

    def make_eval_ds(files, labels=None, cache_name=None):
        opt = tf.data.Options()
        opt.experimental_deterministic = True
        img_ds = make_image_ds(files, cache_name=cache_name)
        if labels is None:
            ds = (
                img_ds.with_options(opt)
                .batch(batch_size, drop_remainder=False)
                .prefetch(AUTO)
            )
            return ds
        lab_ds = tf.data.Dataset.from_tensor_slices(labels)
        ds = tf.data.Dataset.zip((img_ds, lab_ds)).with_options(opt)
        ds = ds.batch(batch_size, drop_remainder=False)
        ds = ds.prefetch(AUTO)
        return ds

    train_ds = make_train_ds(
        train_files, y_train, train_seeds, cache_name="train_images"
    )
    valid_ds = make_eval_ds(val_files, y_val, cache_name="val_images")
    test_ds = make_eval_ds(test_files, labels=None, cache_name="test_images")

    steps_per_epoch = int(np.ceil(len(train_files) / batch_size))
    validation_steps = int(np.ceil(len(val_files) / batch_size))

    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(img_size[0], img_size[1], 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(img_size[0], img_size[1], 3))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.25)(x)
    outputs = tf.keras.layers.Dense(len(label_cols), activation="sigmoid")(x)
    model = tf.keras.Model(inputs, outputs)

    callbacks = [
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
        )
    ]

    # CHANGE (timeout): disable jit_compile to avoid large XLA compile overhead (done twice in original).
    # Semantics are identical; training/inference results stay equivalent up to negligible FP differences.
    model.compile(
        optimizer=Adam(learning_rate=2e-4),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.01),
        metrics=[
            tf.keras.metrics.AUC(
                multi_label=True, num_labels=len(label_cols), name="auc"
            )
        ],
        jit_compile=False,
    )

    # Warm-up removed: with jit_compile=False, there's no expensive compile to trigger here; saves time without changing results.

    model.fit(
        train_ds,
        validation_data=valid_ds,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        epochs=epochs_stage1,
        verbose=1,
        callbacks=callbacks,
    )

    base_model.trainable = True
    for layer in base_model.layers[:-25]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=5e-5),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.01),
        metrics=[
            tf.keras.metrics.AUC(
                multi_label=True, num_labels=len(label_cols), name="auc"
            )
        ],
        jit_compile=False,
    )

    model.fit(
        train_ds,
        validation_data=valid_ds,
        steps_per_epoch=steps_per_epoch,
        validation_steps=validation_steps,
        epochs=epochs_stage2,
        verbose=1,
        callbacks=callbacks,
    )

    p = model.predict(test_ds, verbose=1)
    p = np.clip(p, 0.0, 1.0)

    out = pd.DataFrame(p, columns=label_cols)
    out.insert(0, "image_id", test_image_ids)
    out = out[sample_sub.columns.tolist()]
    return out




In [4]:
isTPU = True

if isTPU:
    df = getPredictionFromTPUModel()
    print(df.head(5))
    df.to_csv("./submission.csv", index=False)
    print("write done -> ./submission.csv")
else:
    df = pd.read_csv("../input/notebook45bc751087/submission.csv")
    df.head()
    result = processAndWriteDf(df)
    result.head(5)

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


2026-05-12 18:39:49.926171: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/tmp/ipykernel_11/676031395.py:214: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/18


2026-05-12 18:39:56.290111: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


47/47 ━━━━━━━━━━━━━━━━━━━━ 51s 783ms/step - auc: 0.5163 - loss: 0.6808 - val_auc: 0.6542 - val_loss: 0.5044 - learning_rate: 2.0000e-04
Epoch 2/18


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


47/47 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - auc: 0.0000e+00 - loss: 0.0000e+00 - val_auc: 0.6542 - val_loss: 0.5044 - learning_rate: 2.0000e-04
Epoch 3/18
47/47 ━━━━━━━━━━━━━━━━━━━━ 37s 758ms/step - auc: 0.5758 - loss: 0.5161 - val_auc: 0.7874 - val_loss: 0.4626 - learning_rate: 2.0000e-04
Epoch 4/18
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step - auc: 0.0000e+00 - loss: 0.0000e+00 - val_auc: 0.7874 - val_loss: 0.4626 - learning_rate: 2.0000e-04
Epoch 5/18
47/47 ━━━━━━━━━━━━━━━━━━━━ 40s 823ms/step - auc: 0.6905 - loss: 0.4665 - val_auc: 0.8589 - val_loss: 0.4281 - learning_rate: 2.0000e-04
Epoch 6/18
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 77ms/step - auc: 0.0000e+00 - loss: 0.0000e+00 - val_auc: 0.8589 - val_loss: 0.4281 - learning_rate: 2.0000e-04
Epoch 7/18
47/47 ━━━━━━━━━━━━━━━━━━━━ 42s 859ms/step - auc: 0.7386 - loss: 0.4360 - val_auc: 0.8895 - val_loss: 0.4014 - learning_rate: 2.0000e-04
Epoch 8/18
47/47 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - auc: 0.0000e+00 - loss: 0.0000e+00 - val_auc: 0.8895 - val